# Medical Staff Scheduling with <8B Open-Weight LLMs
## Deliverable 1 - Feasibility and Baseline Failure Notebook

This notebook demonstrates:
1. **Execution Feasibility** on a free Google Colab T4 GPU (15 GB VRAM).
2. Loading Candidate Model (`Qwen/Qwen2.5-3B-Instruct` or `meta-llama/Llama-3.2-3B-Instruct`) in 4-bit / 16-bit precision.
3. Empirical verification of **Direct Prompting Failure** against the `ScheduleVerifier`.

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Load Model in 4-bit (VRAM consumption ~2.2 GB)
model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

allocated = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded successfully! Allocated VRAM: {allocated:.2f} GB / 15.0 GB (Colab T4)")

In [ ]:
# Step 3: Run Direct Prompting Inference
prompt = """You are an automated medical scheduling assistant.
Assign doctors to shifts for an entire week in valid JSON format without violating:
1. Min 16h rest after Night shifts.
2. Max 40h weekly per doctor.
3. At least 1 Anesthesiologist on Night shifts and 1 Emergency specialist on Morning/Afternoon.
Return JSON only."""

messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=1024, temperature=0.2)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== RAW DIRECT PROMPT OUTPUT ===\n", response)